In [1]:
# ============================================================
# OceanF — ML Dataset Preparation
# Section 1: Imports and Configuration
# ============================================================

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(r"C:\OceanF")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Processed data directory:")
print(PROCESSED_DIR)

Processed data directory:
C:\OceanF\data\processed


In [3]:
# ============================================================
# ML VARIABLE CONFIGURATION
# ============================================================

INPUT_VARIABLES = [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "u_wind",
    "v_wind",
]

TARGET_VARIABLE = "thetao"

TARGET_DEPTHS = np.array([
    0,
    5,
    10,
    20,
    30,
    50,
    75,
    100,
    125,
    150,
    200,
    300,
    500,
    700,
    1000
], dtype=np.float32)

print("Input variables:")
for variable in INPUT_VARIABLES:
    print(" -", variable)

print("\nTarget variable:")
print(" -", TARGET_VARIABLE)

print("\nTarget depths:")
print(TARGET_DEPTHS)

Input variables:
 - sst
 - sss
 - sla
 - uo
 - vo
 - u_wind
 - v_wind

Target variable:
 - thetao

Target depths:
[   0.    5.   10.   20.   30.   50.   75.  100.  125.  150.  200.  300.
  500.  700. 1000.]


In [4]:
# ============================================================
# LIST PROCESSED DATASETS
# ============================================================

for path in sorted(PROCESSED_DIR.rglob("*.nc")):
    print(path)

C:\OceanF\data\processed\Currents\Currents_processed.nc
C:\OceanF\data\processed\SSA\SSA_processed.nc
C:\OceanF\data\processed\SSS\SSS_processed.nc
C:\OceanF\data\processed\SST\SST_processed.nc
C:\OceanF\data\processed\SubsurfaceTemp\SubsurfaceTemp_processed.nc
C:\OceanF\data\processed\Winds\Winds_processed.nc


In [5]:
# ============================================================
# LOAD PROCESSED DATASETS
# ============================================================

DATASET_PATHS = {
    "SST": PROCESSED_DIR / "SST" / "SST_processed.nc",
    "SSS": PROCESSED_DIR / "SSS" / "SSS_processed.nc",
    "SSA": PROCESSED_DIR / "SSA" / "SSA_processed.nc",
    "Currents": PROCESSED_DIR / "Currents" / "Currents_processed.nc",
    "Winds": PROCESSED_DIR / "Winds" / "Winds_processed.nc",
    "SubsurfaceTemp": PROCESSED_DIR / "SubsurfaceTemp" / "SubsurfaceTemp_processed.nc",
}

datasets = {}

for name, path in DATASET_PATHS.items():

    print(f"\nLoading {name}...")
    print(f"Path: {path}")

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    datasets[name] = xr.open_dataset(path)

    print(datasets[name])


Loading SST...
Path: C:\OceanF\data\processed\SST\SST_processed.nc
<xarray.Dataset> Size: 1GB
Dimensions:    (time: 455, latitude: 500, longitude: 1200)
Coordinates:
  * time       (time) datetime64[ns] 4kB 2025-01-01 2025-01-02 ... 2026-03-31
  * latitude   (latitude) float32 2kB 5.025 5.075 5.125 ... 29.88 29.92 29.98
  * longitude  (longitude) float32 5kB 45.03 45.08 45.12 ... 104.9 104.9 105.0
Data variables:
    sst        (time, latitude, longitude) float32 1GB ...
Attributes: (12/48)
    Conventions:                CF-1.4, ACDD-1.3
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    WARNING Some applications are unable to prope...
    creator_email:              servicedesk.cmems@mercator-ocean.eu
    ...                         ...
    time_coverage_end:          2026-03-31T00:00:00
    time_coverage_start:        20

In [7]:
# ============================================================
# TEMPORAL HARMONIZATION FOR ML DATASET
# ============================================================

COMMON_START = "2025-07-01"
COMMON_END = "2025-12-31"

datasets_common = {}

for name, ds in datasets.items():

    # Winds uses valid_time
    if "valid_time" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"valid_time": "time"})

    ds_common = ds.sel(
        time=slice(COMMON_START, COMMON_END)
    )

    datasets_common[name] = ds_common

    print(
        f"{name}: "
        f"{ds_common.sizes['time']} time steps | "
        f"{ds_common.time.values[0]} → "
        f"{ds_common.time.values[-1]}"
    )

SST: 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
SSS: 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
SSA: 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
Currents: 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
Winds: 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
SubsurfaceTemp: 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000


In [9]:
# ============================================================
# DIMENSION CHECK AFTER TEMPORAL HARMONIZATION
# ============================================================

EXPECTED_TIME = 184
EXPECTED_LAT = 101
EXPECTED_LON = 241

for name, ds in datasets_common.items():

    print(f"\n{name}")
    print("-" * 50)

    print("time:", ds.sizes["time"])
    print("latitude:", ds.sizes["latitude"])
    print("longitude:", ds.sizes["longitude"])

    assert ds.sizes["time"] == EXPECTED_TIME, \
        f"{name}: unexpected time dimension"

    assert ds.sizes["latitude"] == EXPECTED_LAT, \
        f"{name}: unexpected latitude dimension"

    assert ds.sizes["longitude"] == EXPECTED_LON, \
        f"{name}: unexpected longitude dimension"

print("\n✅ Temporal dimensions are correct.")
print("✅ Spatial dimensions are correct.")


SST
--------------------------------------------------
time: 184
latitude: 500
longitude: 1200


AssertionError: SST: unexpected latitude dimension

In [10]:
# ============================================================
# TARGET OCEANF GRID
# ============================================================

TARGET_RESOLUTION = 0.25

TARGET_LAT = np.arange(
    5.0,
    30.0 + TARGET_RESOLUTION / 2,
    TARGET_RESOLUTION,
    dtype=np.float32
)

TARGET_LON = np.arange(
    45.0,
    105.0 + TARGET_RESOLUTION / 2,
    TARGET_RESOLUTION,
    dtype=np.float32
)

print("Target latitude:")
print(
    f"  {TARGET_LAT[0]}°N → {TARGET_LAT[-1]}°N"
)
print("  Number of points:", len(TARGET_LAT))

print("\nTarget longitude:")
print(
    f"  {TARGET_LON[0]}°E → {TARGET_LON[-1]}°E"
)
print("  Number of points:", len(TARGET_LON))

print("\nTarget grid:")
print(f"  {len(TARGET_LAT)} × {len(TARGET_LON)}")

assert len(TARGET_LAT) == 101
assert len(TARGET_LON) == 241

print("\n✅ OceanF target grid confirmed.")

Target latitude:
  5.0°N → 30.0°N
  Number of points: 101

Target longitude:
  45.0°E → 105.0°E
  Number of points: 241

Target grid:
  101 × 241

✅ OceanF target grid confirmed.


In [11]:
# ============================================================
# SPATIAL HARMONIZATION
# ============================================================

datasets_harmonized = {}

for name, ds in datasets_common.items():

    print(f"\nHarmonizing {name}...")

    # Ensure latitude is ascending
    if ds.latitude.values[0] > ds.latitude.values[-1]:
        ds = ds.sortby("latitude")

    ds_h = ds.interp(
        latitude=TARGET_LAT,
        longitude=TARGET_LON,
        method="linear",
        kwargs={"fill_value": np.nan}
    )

    datasets_harmonized[name] = ds_h

    print(
        f"  Original: "
        f"{ds.sizes['latitude']} × "
        f"{ds.sizes['longitude']}"
    )

    print(
        f"  Harmonized: "
        f"{ds_h.sizes['latitude']} × "
        f"{ds_h.sizes['longitude']}"
    )


Harmonizing SST...
  Original: 500 × 1200
  Harmonized: 101 × 241

Harmonizing SSS...
  Original: 200 × 480
  Harmonized: 101 × 241

Harmonizing SSA...
  Original: 200 × 480
  Harmonized: 101 × 241

Harmonizing Currents...
  Original: 100 × 240
  Harmonized: 101 × 241

Harmonizing Winds...
  Original: 101 × 241
  Harmonized: 101 × 241

Harmonizing SubsurfaceTemp...
  Original: 101 × 241
  Harmonized: 101 × 241


In [12]:
# ============================================================
# SPATIAL GRID VERIFICATION
# ============================================================

for name, ds in datasets_harmonized.items():

    print(f"\n{name}")
    print("-" * 50)

    lat_match = np.allclose(
        ds.latitude.values,
        TARGET_LAT,
        atol=1e-6
    )

    lon_match = np.allclose(
        ds.longitude.values,
        TARGET_LON,
        atol=1e-6
    )

    lat_ascending = np.all(
        np.diff(ds.latitude.values) > 0
    )

    lon_ascending = np.all(
        np.diff(ds.longitude.values) > 0
    )

    print("Latitude:", lat_match)
    print("Longitude:", lon_match)
    print("Latitude ascending:", lat_ascending)
    print("Longitude ascending:", lon_ascending)

    assert lat_match
    assert lon_match
    assert lat_ascending
    assert lon_ascending

print("\n✅ ALL DATASETS ARE ON THE EXACT OCEANF GRID.")


SST
--------------------------------------------------
Latitude: True
Longitude: True
Latitude ascending: True
Longitude ascending: True

SSS
--------------------------------------------------
Latitude: True
Longitude: True
Latitude ascending: True
Longitude ascending: True

SSA
--------------------------------------------------
Latitude: True
Longitude: True
Latitude ascending: True
Longitude ascending: True

Currents
--------------------------------------------------
Latitude: True
Longitude: True
Latitude ascending: True
Longitude ascending: True

Winds
--------------------------------------------------
Latitude: True
Longitude: True
Latitude ascending: True
Longitude ascending: True

SubsurfaceTemp
--------------------------------------------------
Latitude: True
Longitude: True
Latitude ascending: True
Longitude ascending: True

✅ ALL DATASETS ARE ON THE EXACT OCEANF GRID.


In [15]:
# ============================================================
# FINAL TIME + GRID CONSISTENCY CHECK
# ============================================================

reference = datasets_harmonized["SubsurfaceTemp"]

reference_time = reference.time.values
reference_lat = reference.latitude.values
reference_lon = reference.longitude.values

for name, ds in datasets_harmonized.items():

    print(f"\nChecking {name}...")

    # --------------------------------------------------------
    # TIME
    # --------------------------------------------------------

    time_match = np.array_equal(
        ds.time.values,
        reference_time
    )

    # --------------------------------------------------------
    # SPATIAL GRID
    # --------------------------------------------------------

    lat_match = np.allclose(
        ds.latitude.values,
        reference_lat,
        atol=1e-6
    )

    lon_match = np.allclose(
        ds.longitude.values,
        reference_lon,
        atol=1e-6
    )

    print("  Time:", time_match)
    print("  Latitude:", lat_match)
    print("  Longitude:", lon_match)

    assert time_match, f"{name}: actual time values do not match"
    assert lat_match, f"{name}: latitude values do not match"
    assert lon_match, f"{name}: longitude values do not match"

print("\n" + "=" * 60)
print("✅ ALL TIME COORDINATES MATCH")
print("✅ ALL LATITUDE COORDINATES MATCH")
print("✅ ALL LONGITUDE COORDINATES MATCH")
print("✅ ALL SIX DATASETS ARE FULLY ALIGNED")
print("=" * 60)


Checking SST...
  Time: True
  Latitude: True
  Longitude: True

Checking SSS...
  Time: True
  Latitude: True
  Longitude: True

Checking SSA...
  Time: True
  Latitude: True
  Longitude: True

Checking Currents...
  Time: True
  Latitude: True
  Longitude: True

Checking Winds...
  Time: True
  Latitude: True
  Longitude: True

Checking SubsurfaceTemp...
  Time: True
  Latitude: True
  Longitude: True

✅ ALL TIME COORDINATES MATCH
✅ ALL LATITUDE COORDINATES MATCH
✅ ALL LONGITUDE COORDINATES MATCH
✅ ALL SIX DATASETS ARE FULLY ALIGNED


In [16]:
# ============================================================
# WINDS TIME SANITY CHECK
# ============================================================

print("Winds first date:")
print(datasets_harmonized["Winds"].time.values[0])

print("\nWinds last date:")
print(datasets_harmonized["Winds"].time.values[-1])

print("\nWinds number of dates:")
print(datasets_harmonized["Winds"].sizes["time"])

print("\nReference first date:")
print(reference.time.values[0])

print("\nReference last date:")
print(reference.time.values[-1])

print("\nReference number of dates:")
print(reference.sizes["time"])

Winds first date:
2025-07-01T00:00:00.000000000

Winds last date:
2025-12-31T00:00:00.000000000

Winds number of dates:
184

Reference first date:
2025-07-01T00:00:00.000000000

Reference last date:
2025-12-31T00:00:00.000000000

Reference number of dates:
184


In [17]:
# ============================================================
# ML FEATURE ASSEMBLY
# ============================================================

feature_mapping = {
    "sst": ("SST", "sst"),
    "sss": ("SSS", "sss"),
    "sla": ("SSA", "sla"),
    "uo": ("Currents", "uo"),
    "vo": ("Currents", "vo"),
    "u_wind": ("Winds", "u_wind"),
    "v_wind": ("Winds", "v_wind"),
}

feature_arrays = []

for feature_name, (dataset_name, variable_name) in feature_mapping.items():

    print(f"Adding {feature_name}...")

    da = datasets_harmonized[dataset_name][variable_name]

    # Give every feature a common name
    da = da.rename(feature_name)

    feature_arrays.append(da)

# Combine along a new feature dimension
X = xr.concat(
    feature_arrays,
    dim="feature"
)

# Assign feature names
X = X.assign_coords(
    feature=list(feature_mapping.keys())
)

# Put dimensions in standard ML order
X = X.transpose(
    "time",
    "latitude",
    "longitude",
    "feature"
)

print("\nX created successfully.")
print(X)

Adding sst...
Adding sss...
Adding sla...
Adding uo...
Adding vo...
Adding u_wind...
Adding v_wind...

X created successfully.
<xarray.DataArray 'sst' (time: 184, latitude: 101, longitude: 241, feature: 7)> Size: 251MB
array([[[[            nan,             nan,             nan, ...,
                      nan,  3.17106628e+00,  6.64614105e+00],
         [            nan,             nan,             nan, ...,
                      nan,  3.37101746e+00,  6.53945160e+00],
         [            nan,             nan,             nan, ...,
                      nan,  4.15641785e+00,  6.77626801e+00],
         ...,
         [            nan,             nan,             nan, ...,
                      nan,  1.64152527e+00,  3.86269379e+00],
         [            nan,             nan,             nan, ...,
                      nan,  1.74284363e+00,  3.76479340e+00],
         [            nan,             nan,             nan, ...,
                      nan,  1.89079285e+00,  3.62709808e+00]]

In [18]:
# ============================================================
# VERIFY ML INPUT X
# ============================================================

print("X dimensions:")
print(X.dims)

print("\nX shape:")
print(X.shape)

print("\nFeature names:")
print(X.feature.values)

assert X.sizes["time"] == 184
assert X.sizes["latitude"] == 101
assert X.sizes["longitude"] == 241
assert X.sizes["feature"] == 7

assert list(X.feature.values) == [
    "sst",
    "sss",
    "sla",
    "uo",
    "vo",
    "u_wind",
    "v_wind",
]

print("\n✅ X has the correct shape and feature ordering.")

X dimensions:
('time', 'latitude', 'longitude', 'feature')

X shape:
(184, 101, 241, 7)

Feature names:
['sst' 'sss' 'sla' 'uo' 'vo' 'u_wind' 'v_wind']

✅ X has the correct shape and feature ordering.


In [19]:
# ============================================================
# ML TARGET ASSEMBLY
# ============================================================

Y = datasets_harmonized["SubsurfaceTemp"]["thetao"]

# Ensure standard target dimension order
Y = Y.transpose(
    "time",
    "depth",
    "latitude",
    "longitude"
)

print("Y created successfully.")
print(Y)

Y created successfully.
<xarray.DataArray 'thetao' (time: 184, depth: 15, latitude: 101, longitude: 241)> Size: 269MB
array([[[[      nan,       nan,       nan, ..., 30.306437, 30.21781 ,
          30.20902 ],
         [      nan,       nan,       nan, ..., 30.222939, 30.174597,
          30.182653],
         [      nan,       nan,       nan, ..., 30.12186 , 30.156284,
          30.20609 ],
         ...,
         [      nan,       nan,       nan, ...,       nan,       nan,
                nan],
         [      nan,       nan,       nan, ...,       nan,       nan,
                nan],
         [      nan,       nan,       nan, ...,       nan,       nan,
                nan]],

        [[      nan,       nan,       nan, ..., 30.280388, 30.192493,
          30.185123],
         [      nan,       nan,       nan, ..., 30.187366, 30.145573,
          30.155827],
         [      nan,       nan,       nan, ..., 30.076813, 30.122866,
          30.178532],
...
         [      nan,       nan,   

In [20]:
# ============================================================
# VERIFY ML TARGET Y
# ============================================================

print("Y dimensions:")
print(Y.dims)

print("\nY shape:")
print(Y.shape)

print("\nTarget depths:")
print(Y.depth.values)

assert Y.sizes["time"] == 184
assert Y.sizes["depth"] == 15
assert Y.sizes["latitude"] == 101
assert Y.sizes["longitude"] == 241

assert np.allclose(
    Y.depth.values,
    TARGET_DEPTHS,
    atol=1e-5
)

print("\n✅ Y has the correct shape.")
print("✅ All 15 target depths are correct.")

Y dimensions:
('time', 'depth', 'latitude', 'longitude')

Y shape:
(184, 15, 101, 241)

Target depths:
[   0.    5.   10.   20.   30.   50.   75.  100.  125.  150.  200.  300.
  500.  700. 1000.]

✅ Y has the correct shape.
✅ All 15 target depths are correct.


In [21]:
# ============================================================
# X / Y ALIGNMENT CHECK
# ============================================================

print("Checking X ↔ Y alignment...")

assert np.array_equal(
    X.time.values,
    Y.time.values
)

assert np.allclose(
    X.latitude.values,
    Y.latitude.values,
    atol=1e-6
)

assert np.allclose(
    X.longitude.values,
    Y.longitude.values,
    atol=1e-6
)

print("Time:      ✅")
print("Latitude:  ✅")
print("Longitude: ✅")

print("\n" + "=" * 60)
print("✅ X AND Y ARE PERFECTLY ALIGNED")
print("=" * 60)

Checking X ↔ Y alignment...
Time:      ✅
Latitude:  ✅
Longitude: ✅

✅ X AND Y ARE PERFECTLY ALIGNED


In [22]:
# ============================================================
# INPUT VALIDITY MASK
# ============================================================

# True  = valid observation
# False = missing / unavailable

X_valid_mask = X.notnull()

print("Input validity mask created.")
print(X_valid_mask)

Input validity mask created.
<xarray.DataArray 'sst' (time: 184, latitude: 101, longitude: 241, feature: 7)> Size: 31MB
array([[[[False, False, False, ..., False,  True,  True],
         [False, False, False, ..., False,  True,  True],
         [False, False, False, ..., False,  True,  True],
         ...,
         [False, False, False, ..., False,  True,  True],
         [False, False, False, ..., False,  True,  True],
         [False, False, False, ..., False,  True,  True]],

        [[False, False, False, ..., False,  True,  True],
         [False, False, False, ..., False,  True,  True],
         [False, False, False, ..., False,  True,  True],
         ...,
         [ True,  True,  True, ...,  True,  True,  True],
         [ True,  True,  True, ...,  True,  True,  True],
         [False, False, False, ..., False,  True,  True]],

        [[False, False, False, ..., False,  True,  True],
         [False, False, False, ..., False,  True,  True],
         [False, False, False, ..., 

In [23]:
# ============================================================
# INPUT VALIDITY SUMMARY
# ============================================================

for feature in X.feature.values:

    mask = X_valid_mask.sel(feature=feature)

    valid_fraction = float(
        mask.mean().compute()
    )

    missing_fraction = 1.0 - valid_fraction

    print(
        f"{feature:10s} | "
        f"valid: {valid_fraction * 100:6.2f}% | "
        f"missing: {missing_fraction * 100:6.2f}%"
    )

sst        | valid:  48.17% | missing:  51.83%
sss        | valid:  46.67% | missing:  53.33%
sla        | valid:  48.49% | missing:  51.51%
uo         | valid:  46.31% | missing:  53.69%
vo         | valid:  46.31% | missing:  53.69%
u_wind     | valid: 100.00% | missing:   0.00%
v_wind     | valid: 100.00% | missing:   0.00%


In [24]:
# ============================================================
# TARGET VALIDITY MASK
# ============================================================

Y_valid_mask = Y.notnull()

print("Target validity mask created.")
print(Y_valid_mask)

Target validity mask created.
<xarray.DataArray 'thetao' (time: 184, depth: 15, latitude: 101, longitude: 241)> Size: 67MB
array([[[[False, False, False, ...,  True,  True,  True],
         [False, False, False, ...,  True,  True,  True],
         [False, False, False, ...,  True,  True,  True],
         ...,
         [False, False, False, ..., False, False, False],
         [False, False, False, ..., False, False, False],
         [False, False, False, ..., False, False, False]],

        [[False, False, False, ...,  True,  True,  True],
         [False, False, False, ...,  True,  True,  True],
         [False, False, False, ...,  True,  True,  True],
         ...,
         [False, False, False, ..., False, False, False],
         [False, False, False, ..., False, False, False],
         [False, False, False, ..., False, False, False]],

        [[False, False, False, ...,  True,  True,  True],
         [False, False, False, ...,  True,  True,  True],
         [False, False, False, ..

In [25]:
# ============================================================
# TARGET VALIDITY BY DEPTH
# ============================================================

print("Target valid fraction by depth:")
print("-" * 45)

for depth in Y.depth.values:

    mask = Y_valid_mask.sel(depth=depth)

    valid_fraction = float(
        mask.mean().compute()
    )

    print(
        f"{float(depth):7.1f} m | "
        f"valid: {valid_fraction * 100:6.2f}% | "
        f"missing: {(1-valid_fraction) * 100:6.2f}%"
    )

Target valid fraction by depth:
---------------------------------------------
    0.0 m | valid:  47.55% | missing:  52.45%
    5.0 m | valid:  47.55% | missing:  52.45%
   10.0 m | valid:  46.13% | missing:  53.87%
   20.0 m | valid:  45.04% | missing:  54.96%
   30.0 m | valid:  43.76% | missing:  56.24%
   50.0 m | valid:  41.68% | missing:  58.32%
   75.0 m | valid:  40.21% | missing:  59.79%
  100.0 m | valid:  39.35% | missing:  60.65%
  125.0 m | valid:  39.19% | missing:  60.81%
  150.0 m | valid:  38.93% | missing:  61.07%
  200.0 m | valid:  38.52% | missing:  61.48%
  300.0 m | valid:  38.20% | missing:  61.80%
  500.0 m | valid:  37.49% | missing:  62.51%
  700.0 m | valid:  36.89% | missing:  63.11%
 1000.0 m | valid:  35.82% | missing:  64.18%


In [26]:
# ============================================================
# CHECK FOR COMPLETELY MISSING FEATURES / TARGET DEPTHS
# ============================================================

print("Checking input features...")

for feature in X.feature.values:

    mask = X_valid_mask.sel(feature=feature)

    valid_count = int(
        mask.sum().compute()
    )

    print(
        f"{feature:10s} | "
        f"valid values: {valid_count:,}"
    )

    assert valid_count > 0, \
        f"{feature} contains no valid observations"


print("\nChecking target depths...")

for depth in Y.depth.values:

    mask = Y_valid_mask.sel(depth=depth)

    valid_count = int(
        mask.sum().compute()
    )

    print(
        f"{float(depth):7.1f} m | "
        f"valid values: {valid_count:,}"
    )

    assert valid_count > 0, \
        f"Depth {depth} contains no valid observations"


print("\n✅ Every input feature has valid observations.")
print("✅ Every target depth has valid observations.")

Checking input features...
sst        | valid values: 2,157,584
sss        | valid values: 2,090,240
sla        | valid values: 2,171,752
uo         | valid values: 2,073,985
vo         | valid values: 2,073,905
u_wind     | valid values: 4,478,744
v_wind     | valid values: 4,478,744

Checking target depths...
    0.0 m | valid values: 2,129,616
    5.0 m | valid values: 2,129,616
   10.0 m | valid values: 2,066,136
   20.0 m | valid values: 2,017,192
   30.0 m | valid values: 1,959,784
   50.0 m | valid values: 1,866,680
   75.0 m | valid values: 1,800,808
  100.0 m | valid values: 1,762,536
  125.0 m | valid values: 1,755,176
  150.0 m | valid values: 1,743,400
  200.0 m | valid values: 1,725,368
  300.0 m | valid values: 1,710,832
  500.0 m | valid values: 1,679,184
  700.0 m | valid values: 1,652,136
 1000.0 m | valid values: 1,604,480

✅ Every input feature has valid observations.
✅ Every target depth has valid observations.


In [27]:
# ============================================================
# STEP 21 — Define chronological train / validation / test split
# ============================================================

TRAIN_START = "2025-07-01"
TRAIN_END   = "2025-10-31"

VAL_START = "2025-11-01"
VAL_END   = "2025-11-30"

TEST_START = "2025-12-01"
TEST_END   = "2025-12-31"

train_mask = (X.time >= TRAIN_START) & (X.time <= TRAIN_END)
val_mask   = (X.time >= VAL_START) & (X.time <= VAL_END)
test_mask  = (X.time >= TEST_START) & (X.time <= TEST_END)

print("Train:", train_mask.sum().compute().item(), "days")
print("Validation:", val_mask.sum().compute().item(), "days")
print("Test:", test_mask.sum().compute().item(), "days")

assert train_mask.sum().compute().item() > 0
assert val_mask.sum().compute().item() > 0
assert test_mask.sum().compute().item() > 0

UFuncTypeError: ufunc 'greater_equal' did not contain a loop with signature matching types (<class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.StrDType'>) -> None

In [28]:
# ============================================================
# STEP 21 — Define chronological train / validation / test split
# ============================================================

TRAIN_START = np.datetime64("2025-07-01")
TRAIN_END   = np.datetime64("2025-10-31")

VAL_START = np.datetime64("2025-11-01")
VAL_END   = np.datetime64("2025-11-30")

TEST_START = np.datetime64("2025-12-01")
TEST_END   = np.datetime64("2025-12-31")

train_mask = (X.time >= TRAIN_START) & (X.time <= TRAIN_END)
val_mask   = (X.time >= VAL_START) & (X.time <= VAL_END)
test_mask  = (X.time >= TEST_START) & (X.time <= TEST_END)

print("Train:", train_mask.sum().compute().item(), "days")
print("Validation:", val_mask.sum().compute().item(), "days")
print("Test:", test_mask.sum().compute().item(), "days")

Train: 123 days
Validation: 30 days
Test: 31 days


In [29]:
# ============================================================
# STEP 22 — Verify split boundaries
# ============================================================

train_times = X.time.where(train_mask, drop=True).values
val_times   = X.time.where(val_mask, drop=True).values
test_times  = X.time.where(test_mask, drop=True).values

print("Train:", train_times[0], "→", train_times[-1])
print("Val:  ", val_times[0],   "→", val_times[-1])
print("Test: ", test_times[0],  "→", test_times[-1])

assert train_times[-1] < val_times[0]
assert val_times[-1] < test_times[0]

assert len(set(train_times) & set(val_times)) == 0
assert len(set(train_times) & set(test_times)) == 0
assert len(set(val_times) & set(test_times)) == 0

print("\n✓ No temporal overlap between train / validation / test")

Train: 2025-07-01T00:00:00.000000000 → 2025-10-31T00:00:00.000000000
Val:   2025-11-01T00:00:00.000000000 → 2025-11-30T00:00:00.000000000
Test:  2025-12-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000

✓ No temporal overlap between train / validation / test


In [30]:
# ============================================================
# STEP 23 — Create X/Y train, validation, test datasets
# ============================================================

X_train = X.sel(time=slice(TRAIN_START, TRAIN_END))
X_val   = X.sel(time=slice(VAL_START, VAL_END))
X_test  = X.sel(time=slice(TEST_START, TEST_END))

Y_train = Y.sel(time=slice(TRAIN_START, TRAIN_END))
Y_val   = Y.sel(time=slice(VAL_START, VAL_END))
Y_test  = Y.sel(time=slice(TEST_START, TEST_END))

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("Y_train:", Y_train.shape)
print("Y_val:  ", Y_val.shape)
print("Y_test: ", Y_test.shape)

X_train: (123, 101, 241, 7)
X_val:   (30, 101, 241, 7)
X_test:  (31, 101, 241, 7)
Y_train: (123, 15, 101, 241)
Y_val:   (30, 15, 101, 241)
Y_test:  (31, 15, 101, 241)


In [31]:
# ============================================================
# STEP 24 — Verify validity coverage in each split
# ============================================================

print("INPUT VALID COVERAGE")
print("=" * 60)

for feature in X.feature.values:
    train_valid = float(X_train.sel(feature=feature).notnull().mean().compute())
    val_valid   = float(X_val.sel(feature=feature).notnull().mean().compute())
    test_valid  = float(X_test.sel(feature=feature).notnull().mean().compute())

    print(
        f"{feature:10s} | "
        f"Train: {train_valid*100:6.2f}% | "
        f"Val: {val_valid*100:6.2f}% | "
        f"Test: {test_valid*100:6.2f}%"
    )

INPUT VALID COVERAGE
sst        | Train:  48.17% | Val:  48.17% | Test:  48.17%
sss        | Train:  46.67% | Val:  46.67% | Test:  46.67%
sla        | Train:  48.49% | Val:  48.49% | Test:  48.49%
uo         | Train:  46.31% | Val:  46.31% | Test:  46.31%
vo         | Train:  46.30% | Val:  46.31% | Test:  46.31%
u_wind     | Train: 100.00% | Val: 100.00% | Test: 100.00%
v_wind     | Train: 100.00% | Val: 100.00% | Test: 100.00%


In [32]:
print("\nTARGET VALID COVERAGE BY DEPTH")
print("=" * 60)

for depth in Y.depth.values:
    train_valid = float(
        Y_train.sel(depth=depth).notnull().mean().compute()
    )
    val_valid = float(
        Y_val.sel(depth=depth).notnull().mean().compute()
    )
    test_valid = float(
        Y_test.sel(depth=depth).notnull().mean().compute()
    )

    print(
        f"{float(depth):7.1f} m | "
        f"Train: {train_valid*100:6.2f}% | "
        f"Val: {val_valid*100:6.2f}% | "
        f"Test: {test_valid*100:6.2f}%"
    )


TARGET VALID COVERAGE BY DEPTH
    0.0 m | Train:  47.55% | Val:  47.55% | Test:  47.55%
    5.0 m | Train:  47.55% | Val:  47.55% | Test:  47.55%
   10.0 m | Train:  46.13% | Val:  46.13% | Test:  46.13%
   20.0 m | Train:  45.04% | Val:  45.04% | Test:  45.04%
   30.0 m | Train:  43.76% | Val:  43.76% | Test:  43.76%
   50.0 m | Train:  41.68% | Val:  41.68% | Test:  41.68%
   75.0 m | Train:  40.21% | Val:  40.21% | Test:  40.21%
  100.0 m | Train:  39.35% | Val:  39.35% | Test:  39.35%
  125.0 m | Train:  39.19% | Val:  39.19% | Test:  39.19%
  150.0 m | Train:  38.93% | Val:  38.93% | Test:  38.93%
  200.0 m | Train:  38.52% | Val:  38.52% | Test:  38.52%
  300.0 m | Train:  38.20% | Val:  38.20% | Test:  38.20%
  500.0 m | Train:  37.49% | Val:  37.49% | Test:  37.49%
  700.0 m | Train:  36.89% | Val:  36.89% | Test:  36.89%
 1000.0 m | Train:  35.82% | Val:  35.82% | Test:  35.82%


In [33]:
# ============================================================
# STEP 25A — Calculate training-only normalization statistics
# ============================================================

feature_stats = {}

print("TRAINING NORMALIZATION STATISTICS")
print("=" * 75)

for feature in X.feature.values:

    da = X_train.sel(feature=feature)

    mean = float(da.mean(skipna=True).compute())
    std  = float(da.std(skipna=True).compute())
    count = int(da.count().compute())

    feature_stats[str(feature)] = {
        "mean": mean,
        "std": std,
        "valid_count": count
    }

    print(
        f"{feature:10s} | "
        f"mean = {mean:12.6f} | "
        f"std = {std:12.6f} | "
        f"valid = {count:,}"
    )

TRAINING NORMALIZATION STATISTICS
sst        | mean =    28.404751 | std =     1.742726 | valid = 1,442,298
sss        | mean =    34.659655 | std =     2.189792 | valid = 1,397,280
sla        | mean =     0.127381 | std =     0.107240 | valid = 1,451,769
uo         | mean =     0.158851 | std =     0.293742 | valid = 1,386,393
vo         | mean =     0.004651 | std =     0.266965 | valid = 1,386,313
u_wind     | mean =     2.492416 | std =     3.554870 | valid = 2,993,943
v_wind     | mean =     1.666000 | std =     3.334402 | valid = 2,993,943


In [34]:
# ============================================================
# STEP 25B — Validate normalization statistics
# ============================================================

for feature, stats in feature_stats.items():

    assert np.isfinite(stats["mean"]), f"{feature}: mean is invalid"
    assert np.isfinite(stats["std"]), f"{feature}: std is invalid"
    assert stats["std"] > 0, f"{feature}: std must be > 0"
    assert stats["valid_count"] > 0, f"{feature}: no valid observations"

print("✓ All training normalization statistics are valid")

✓ All training normalization statistics are valid


In [35]:
# ============================================================
# STEP 26A — Training-only target statistics by depth
# ============================================================

target_stats = {}

print("TARGET NORMALIZATION STATISTICS")
print("=" * 80)

for depth in Y_train.depth.values:

    da = Y_train.sel(depth=depth)

    mean = float(da.mean(skipna=True).compute())
    std = float(da.std(skipna=True).compute())
    count = int(da.count().compute())

    target_stats[float(depth)] = {
        "mean": mean,
        "std": std,
        "valid_count": count
    }

    print(
        f"{float(depth):7.1f} m | "
        f"mean = {mean:10.4f} °C | "
        f"std = {std:10.4f} °C | "
        f"valid = {count:,}"
    )

TARGET NORMALIZATION STATISTICS
    0.0 m | mean =    28.4865 °C | std =     1.6479 °C | valid = 1,423,602
    5.0 m | mean =    28.4393 °C | std =     1.6437 °C | valid = 1,423,602
   10.0 m | mean =    28.3596 °C | std =     1.6249 °C | valid = 1,381,167
   20.0 m | mean =    28.1954 °C | std =     1.6808 °C | valid = 1,348,449
   30.0 m | mean =    27.9291 °C | std =     1.8577 °C | valid = 1,310,073
   50.0 m | mean =    27.1992 °C | std =     2.3706 °C | valid = 1,247,835
   75.0 m | mean =    25.8876 °C | std =     2.7070 °C | valid = 1,203,801
  100.0 m | mean =    23.8238 °C | std =     2.5781 °C | valid = 1,178,217
  125.0 m | mean =    21.2263 °C | std =     2.3344 °C | valid = 1,173,297
  150.0 m | mean =    18.8628 °C | std =     2.2079 °C | valid = 1,165,425
  200.0 m | mean =    15.8661 °C | std =     1.9165 °C | valid = 1,153,371
  300.0 m | mean =    13.0932 °C | std =     1.3613 °C | valid = 1,143,654
  500.0 m | mean =    11.2242 °C | std =     1.1399 °C | valid = 1,1

In [36]:
# ============================================================
# STEP 26B — Validate target statistics
# ============================================================

for depth, stats in target_stats.items():

    assert np.isfinite(stats["mean"]), \
        f"{depth} m: mean is invalid"

    assert np.isfinite(stats["std"]), \
        f"{depth} m: std is invalid"

    assert stats["std"] > 0, \
        f"{depth} m: std must be > 0"

    assert stats["valid_count"] > 0, \
        f"{depth} m: no valid observations"

print("✓ All target-depth normalization statistics are valid")

✓ All target-depth normalization statistics are valid


In [37]:
# ============================================================
# STEP 26C — Compact target climatology table
# ============================================================

print("\nDepth-wise training temperature statistics")
print("-" * 50)

for depth in Y_train.depth.values:
    stats = target_stats[float(depth)]

    print(
        f"{float(depth):6.1f} m : "
        f"{stats['mean']:7.3f} °C ± "
        f"{stats['std']:6.3f} °C"
    )


Depth-wise training temperature statistics
--------------------------------------------------
   0.0 m :  28.486 °C ±  1.648 °C
   5.0 m :  28.439 °C ±  1.644 °C
  10.0 m :  28.360 °C ±  1.625 °C
  20.0 m :  28.195 °C ±  1.681 °C
  30.0 m :  27.929 °C ±  1.858 °C
  50.0 m :  27.199 °C ±  2.371 °C
  75.0 m :  25.888 °C ±  2.707 °C
 100.0 m :  23.824 °C ±  2.578 °C
 125.0 m :  21.226 °C ±  2.334 °C
 150.0 m :  18.863 °C ±  2.208 °C
 200.0 m :  15.866 °C ±  1.917 °C
 300.0 m :  13.093 °C ±  1.361 °C
 500.0 m :  11.224 °C ±  1.140 °C
 700.0 m :   9.769 °C ±  1.191 °C
1000.0 m :   7.705 °C ±  1.053 °C


In [38]:
# ============================================================
# STEP 27A — Normalize input features using TRAIN statistics
# ============================================================

X_train_norm = X_train.copy()
X_val_norm   = X_val.copy()
X_test_norm  = X_test.copy()

for feature in X.feature.values:

    feature = str(feature)

    mean = feature_stats[feature]["mean"]
    std  = feature_stats[feature]["std"]

    X_train_norm.loc[dict(feature=feature)] = (
        X_train.sel(feature=feature) - mean
    ) / std

    X_val_norm.loc[dict(feature=feature)] = (
        X_val.sel(feature=feature) - mean
    ) / std

    X_test_norm.loc[dict(feature=feature)] = (
        X_test.sel(feature=feature) - mean
    ) / std

print("✓ Input normalization applied using training statistics only")

✓ Input normalization applied using training statistics only


In [39]:
# ============================================================
# STEP 27B — Verify normalized training inputs
# ============================================================

print("NORMALIZED TRAINING INPUTS")
print("=" * 65)

for feature in X_train_norm.feature.values:

    da = X_train_norm.sel(feature=feature)

    mean = float(da.mean(skipna=True).compute())
    std  = float(da.std(skipna=True).compute())

    print(
        f"{str(feature):10s} | "
        f"mean = {mean:10.5f} | "
        f"std = {std:10.5f}"
    )

NORMALIZED TRAINING INPUTS
sst        | mean =   -0.00000 | std =    1.00000
sss        | mean =   -0.00000 | std =    1.00000
sla        | mean =    0.00000 | std =    1.00000
uo         | mean =    0.00000 | std =    1.00000
vo         | mean =    0.00000 | std =    1.00000
u_wind     | mean =    0.00000 | std =    1.00000
v_wind     | mean =    0.00000 | std =    1.00000


In [40]:
# ============================================================
# STEP 28A — Normalize target temperature by depth
# ============================================================

Y_train_norm = Y_train.copy()
Y_val_norm   = Y_val.copy()
Y_test_norm  = Y_test.copy()

for depth in Y.depth.values:

    depth_key = float(depth)

    mean = target_stats[depth_key]["mean"]
    std  = target_stats[depth_key]["std"]

    Y_train_norm.loc[dict(depth=depth)] = (
        Y_train.sel(depth=depth) - mean
    ) / std

    Y_val_norm.loc[dict(depth=depth)] = (
        Y_val.sel(depth=depth) - mean
    ) / std

    Y_test_norm.loc[dict(depth=depth)] = (
        Y_test.sel(depth=depth) - mean
    ) / std

print("✓ Target normalization applied depth-by-depth")
print("✓ Training statistics used for all three splits")
print("✓ Target NaNs preserved")

✓ Target normalization applied depth-by-depth
✓ Training statistics used for all three splits
✓ Target NaNs preserved


In [41]:
# ============================================================
# STEP 28B — Verify normalized training targets
# ============================================================

print("NORMALIZED TRAINING TARGET")
print("=" * 70)

for depth in Y_train_norm.depth.values:

    da = Y_train_norm.sel(depth=depth)

    mean = float(da.mean(skipna=True).compute())
    std = float(da.std(skipna=True).compute())

    print(
        f"{float(depth):7.1f} m | "
        f"mean = {mean:10.5f} | "
        f"std = {std:10.5f}"
    )

NORMALIZED TRAINING TARGET
    0.0 m | mean =   -0.00000 | std =    1.00000
    5.0 m | mean =   -0.00000 | std =    1.00000
   10.0 m | mean =    0.00000 | std =    1.00000
   20.0 m | mean =   -0.00000 | std =    1.00000
   30.0 m | mean =    0.00000 | std =    1.00000
   50.0 m | mean =   -0.00000 | std =    1.00000
   75.0 m | mean =   -0.00000 | std =    1.00000
  100.0 m | mean =    0.00000 | std =    1.00000
  125.0 m | mean =    0.00000 | std =    1.00000
  150.0 m | mean =   -0.00000 | std =    1.00000
  200.0 m | mean =    0.00000 | std =    1.00000
  300.0 m | mean =    0.00000 | std =    1.00000
  500.0 m | mean =   -0.00000 | std =    1.00000
  700.0 m | mean =    0.00000 | std =    1.00000
 1000.0 m | mean =   -0.00000 | std =    1.00000


In [42]:
# ============================================================
# STEP 29 — Verify missing-data structure is unchanged
# ============================================================

for feature in X.feature.values:

    original_count = int(
        X_train.sel(feature=feature).count().compute()
    )

    normalized_count = int(
        X_train_norm.sel(feature=feature).count().compute()
    )

    assert original_count == normalized_count, (
        f"{feature}: valid-count changed after normalization"
    )

for depth in Y.depth.values:

    original_count = int(
        Y_train.sel(depth=depth).count().compute()
    )

    normalized_count = int(
        Y_train_norm.sel(depth=depth).count().compute()
    )

    assert original_count == normalized_count, (
        f"{float(depth)} m: valid-count changed after normalization"
    )

print("✓ Input NaN structure preserved")
print("✓ Target NaN structure preserved")

✓ Input NaN structure preserved
✓ Target NaN structure preserved


In [43]:
# ============================================================
# STEP 30A — Create input validity masks
# ============================================================

X_train_mask = X_train_norm.notnull()
X_val_mask   = X_val_norm.notnull()
X_test_mask  = X_test_norm.notnull()

print("✓ Input validity masks created")

print("X_train_mask shape:", X_train_mask.shape)
print("X_val_mask shape:  ", X_val_mask.shape)
print("X_test_mask shape: ", X_test_mask.shape)

✓ Input validity masks created
X_train_mask shape: (123, 101, 241, 7)
X_val_mask shape:   (30, 101, 241, 7)
X_test_mask shape:  (31, 101, 241, 7)


In [44]:
# ============================================================
# STEP 30B — Create target validity masks
# ============================================================

Y_train_mask = Y_train_norm.notnull()
Y_val_mask   = Y_val_norm.notnull()
Y_test_mask  = Y_test_norm.notnull()

print("✓ Target validity masks created")

print("Y_train_mask shape:", Y_train_mask.shape)
print("Y_val_mask shape:  ", Y_val_mask.shape)
print("Y_test_mask shape: ", Y_test_mask.shape)

✓ Target validity masks created
Y_train_mask shape: (123, 15, 101, 241)
Y_val_mask shape:   (30, 15, 101, 241)
Y_test_mask shape:  (31, 15, 101, 241)


In [45]:
# ============================================================
# STEP 31 — Convert masks to float32
# ============================================================

X_train_mask = X_train_mask.astype(np.float32)
X_val_mask   = X_val_mask.astype(np.float32)
X_test_mask  = X_test_mask.astype(np.float32)

Y_train_mask = Y_train_mask.astype(np.float32)
Y_val_mask   = Y_val_mask.astype(np.float32)
Y_test_mask  = Y_test_mask.astype(np.float32)

print("✓ Masks converted to float32")

✓ Masks converted to float32


In [46]:
# ============================================================
# STEP 32 — Verify masks contain only 0 and 1
# ============================================================

for name, mask in {
    "X_train": X_train_mask,
    "X_val": X_val_mask,
    "X_test": X_test_mask,
    "Y_train": Y_train_mask,
    "Y_val": Y_val_mask,
    "Y_test": Y_test_mask,
}.items():

    values = np.unique(mask.values)

    print(f"{name:10s}:", values)

    assert np.all(np.isin(values, [0.0, 1.0]))

print("\n✓ All masks contain only 0.0 and 1.0")

X_train   : [0. 1.]
X_val     : [0. 1.]
X_test    : [0. 1.]
Y_train   : [0. 1.]
Y_val     : [0. 1.]
Y_test    : [0. 1.]

✓ All masks contain only 0.0 and 1.0


In [47]:
# ============================================================
# STEP 33A — Convert X to ML channel-first layout
# ============================================================

X_train_ml = X_train_norm.transpose(
    "time", "feature", "latitude", "longitude"
)

X_val_ml = X_val_norm.transpose(
    "time", "feature", "latitude", "longitude"
)

X_test_ml = X_test_norm.transpose(
    "time", "feature", "latitude", "longitude"
)

print("X_train_ml:", X_train_ml.shape)
print("X_val_ml:  ", X_val_ml.shape)
print("X_test_ml: ", X_test_ml.shape)

X_train_ml: (123, 7, 101, 241)
X_val_ml:   (30, 7, 101, 241)
X_test_ml:  (31, 7, 101, 241)


In [48]:
# ============================================================
# STEP 33B — Confirm Y channel/depth-first layout
# ============================================================

Y_train_ml = Y_train_norm.transpose(
    "time", "depth", "latitude", "longitude"
)

Y_val_ml = Y_val_norm.transpose(
    "time", "depth", "latitude", "longitude"
)

Y_test_ml = Y_test_norm.transpose(
    "time", "depth", "latitude", "longitude"
)

print("Y_train_ml:", Y_train_ml.shape)
print("Y_val_ml:  ", Y_val_ml.shape)
print("Y_test_ml: ", Y_test_ml.shape)

Y_train_ml: (123, 15, 101, 241)
Y_val_ml:   (30, 15, 101, 241)
Y_test_ml:  (31, 15, 101, 241)


In [49]:
# ============================================================
# STEP 33C — ML layouts for validity masks
# ============================================================

X_train_mask_ml = X_train_mask.transpose(
    "time", "feature", "latitude", "longitude"
)

X_val_mask_ml = X_val_mask.transpose(
    "time", "feature", "latitude", "longitude"
)

X_test_mask_ml = X_test_mask.transpose(
    "time", "feature", "latitude", "longitude"
)

Y_train_mask_ml = Y_train_mask.transpose(
    "time", "depth", "latitude", "longitude"
)

Y_val_mask_ml = Y_val_mask.transpose(
    "time", "depth", "latitude", "longitude"
)

Y_test_mask_ml = Y_test_mask.transpose(
    "time", "depth", "latitude", "longitude"
)

print("✓ ML layouts established for data and masks")

✓ ML layouts established for data and masks


In [50]:
# ============================================================
# STEP 33D — Verify ML dimensions and coordinates
# ============================================================

assert X_train_ml.dims == ("time", "feature", "latitude", "longitude")
assert Y_train_ml.dims == ("time", "depth", "latitude", "longitude")

assert X_train_mask_ml.shape == X_train_ml.shape
assert Y_train_mask_ml.shape == Y_train_ml.shape

assert X_train_ml.feature.values.tolist() == [
    "sst", "sss", "sla", "uo", "vo", "u_wind", "v_wind"
]

assert np.allclose(
    Y_train_ml.depth.values,
    TARGET_DEPTHS,
    atol=1e-6
)

print("✓ X/Y/mask dimensions aligned")
print("✓ Feature order verified")
print("✓ Target depth order verified")

✓ X/Y/mask dimensions aligned
✓ Feature order verified
✓ Target depth order verified


In [51]:
# ============================================================
# STEP 34A — Replace missing normalized X values with 0
# ============================================================

X_train_ml_filled = X_train_ml.fillna(0.0)
X_val_ml_filled   = X_val_ml.fillna(0.0)
X_test_ml_filled  = X_test_ml.fillna(0.0)

print("✓ Missing normalized input values replaced with 0.0")

✓ Missing normalized input values replaced with 0.0


In [52]:
# ============================================================
# STEP 34B — Verify X contains no NaNs
# ============================================================

train_nan = int(X_train_ml_filled.isnull().sum().compute())
val_nan   = int(X_val_ml_filled.isnull().sum().compute())
test_nan  = int(X_test_ml_filled.isnull().sum().compute())

print("Train NaNs:", train_nan)
print("Val NaNs:  ", val_nan)
print("Test NaNs: ", test_nan)

assert train_nan == 0
assert val_nan == 0
assert test_nan == 0

print("\n✓ No NaNs remain in ML input tensors")

Train NaNs: 0
Val NaNs:   0
Test NaNs:  0

✓ No NaNs remain in ML input tensors


In [54]:
# ============================================================
# STEP 34C — Verify filled values match the validity masks
# ============================================================

for X_data, X_mask, name in [
    (X_train_ml_filled, X_train_mask_ml, "Train"),
    (X_val_ml_filled, X_val_mask_ml, "Validation"),
    (X_test_ml_filled, X_test_mask_ml, "Test"),
]:

    # Locations that were originally missing
    missing = (X_mask == 0)

    # At those locations, filled X must equal 0
    wrong = (X_data != 0) & missing

    wrong_count = int(wrong.sum().compute())

    print(f"{name:12s} | incorrect filled values: {wrong_count:,}")

    assert wrong_count == 0

print("\n✓ Filled input values are consistent with validity masks")

Train        | incorrect filled values: 0
Validation   | incorrect filled values: 0
Test         | incorrect filled values: 0

✓ Filled input values are consistent with validity masks


In [55]:
# ============================================================
# STEP 35 — Estimate ML tensor sizes
# ============================================================

def tensor_size_gb(shape, dtype_bytes=4):
    return np.prod(shape) * dtype_bytes / (1024**3)

print("Estimated float32 sizes")
print("=" * 55)

print(
    f"X train: {tensor_size_gb(X_train_ml.shape):.3f} GB"
)

print(
    f"Y train: {tensor_size_gb(Y_train_ml.shape):.3f} GB"
)

print(
    f"X val:   {tensor_size_gb(X_val_ml.shape):.3f} GB"
)

print(
    f"Y val:   {tensor_size_gb(Y_val_ml.shape):.3f} GB"
)

print(
    f"X test:  {tensor_size_gb(X_test_ml.shape):.3f} GB"
)

print(
    f"Y test:  {tensor_size_gb(Y_test_ml.shape):.3f} GB"
)

Estimated float32 sizes
X train: 0.078 GB
Y train: 0.167 GB
X val:   0.019 GB
Y val:   0.041 GB
X test:  0.020 GB
Y test:  0.042 GB


In [56]:
# Step 35: Check ML dataset memory footprint
import numpy as np

def memory_mb(da):
    """
    Estimate memory required if the DataArray is materialized
    as float32/its current dtype.
    """
    n_elements = int(np.prod(da.shape))
    return n_elements * np.dtype(da.dtype).itemsize / (1024 ** 2)


arrays_to_check = {
    "X_train": X_train_ml_filled,
    "X_val": X_val_ml_filled,
    "X_test": X_test_ml_filled,

    "Y_train": Y_train_ml,
    "Y_val": Y_val_ml,
    "Y_test": Y_test_ml,

    "X_train_mask": X_train_mask_ml,
    "X_val_mask": X_val_mask_ml,
    "X_test_mask": X_test_mask_ml,

    "Y_train_mask": Y_train_mask_ml,
    "Y_val_mask": Y_val_mask_ml,
    "Y_test_mask": Y_test_mask_ml,
}

print("ML dataset memory estimate")
print("=" * 65)

total_mb = 0.0

for name, da in arrays_to_check.items():
    mb = memory_mb(da)
    total_mb += mb

    print(
        f"{name:16s} | "
        f"shape={str(da.shape):28s} | "
        f"dtype={str(da.dtype):8s} | "
        f"~{mb:8.2f} MB"
    )

print("=" * 65)
print(f"Estimated total if all arrays are materialized: {total_mb:.2f} MB")
print(f"Estimated total: {total_mb / 1024:.2f} GB")

ML dataset memory estimate
X_train          | shape=(123, 7, 101, 241)           | dtype=float64  | ~  159.89 MB
X_val            | shape=(30, 7, 101, 241)            | dtype=float64  | ~   39.00 MB
X_test           | shape=(31, 7, 101, 241)            | dtype=float64  | ~   40.30 MB
Y_train          | shape=(123, 15, 101, 241)          | dtype=float32  | ~  171.31 MB
Y_val            | shape=(30, 15, 101, 241)           | dtype=float32  | ~   41.78 MB
Y_test           | shape=(31, 15, 101, 241)           | dtype=float32  | ~   43.18 MB
X_train_mask     | shape=(123, 7, 101, 241)           | dtype=float32  | ~   79.95 MB
X_val_mask       | shape=(30, 7, 101, 241)            | dtype=float32  | ~   19.50 MB
X_test_mask      | shape=(31, 7, 101, 241)            | dtype=float32  | ~   20.15 MB
Y_train_mask     | shape=(123, 15, 101, 241)          | dtype=float32  | ~  171.31 MB
Y_val_mask       | shape=(30, 15, 101, 241)           | dtype=float32  | ~   41.78 MB
Y_test_mask      | shape=(3

In [57]:
# Check whether the arrays are still lazily represented
for name, da in arrays_to_check.items():
    print(
        f"{name:16s} | "
        f"backend={type(da.data).__name__}"
    )

X_train          | backend=ndarray
X_val            | backend=ndarray
X_test           | backend=ndarray
Y_train          | backend=ndarray
Y_val            | backend=ndarray
Y_test           | backend=ndarray
X_train_mask     | backend=ndarray
X_val_mask       | backend=ndarray
X_test_mask      | backend=ndarray
Y_train_mask     | backend=ndarray
Y_val_mask       | backend=ndarray
Y_test_mask      | backend=ndarray


In [58]:
# Step 36: Save ML configuration and normalization statistics

import json
from pathlib import Path

ML_CONFIG_DIR = PROJECT_ROOT / "data" / "processed" / "ML"
ML_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "project": "OceanF",

    "domain": {
        "latitude_min": 5.0,
        "latitude_max": 30.0,
        "longitude_min": 45.0,
        "longitude_max": 105.0,
        "resolution": 0.25
    },

    "time": {
        "common_start": "2025-07-01",
        "common_end": "2025-12-31",

        "train_start": "2025-07-01",
        "train_end": "2025-10-31",

        "validation_start": "2025-11-01",
        "validation_end": "2025-11-30",

        "test_start": "2025-12-01",
        "test_end": "2025-12-31"
    },

    "input_features": [
        "sst",
        "sss",
        "sla",
        "uo",
        "vo",
        "u_wind",
        "v_wind"
    ],

    "target_variable": "thetao",

    "target_depths_m": [
        0, 5, 10, 20, 30,
        50, 75, 100, 125, 150,
        200, 300, 500, 700, 1000
    ],

    "normalization": {
        "inputs": "training_period_mean_std",
        "target": "training_period_mean_std_by_depth"
    },

    "missing_data": {
        "input_missing_values": "filled_with_zero_after_normalization",
        "input_validity_mask": "retained",
        "target_missing_values": "retained",
        "target_validity_mask": "retained",
        "target_loss": "masked"
    },

    "spatial_harmonization": {
        "method": "linear_interpolation",
        "target_resolution": 0.25
    },

    "split_strategy": "chronological"
}

# Add normalization statistics
config["input_statistics"] = feature_stats
config["target_statistics"] = {
    str(depth): stats
    for depth, stats in target_stats.items()
}

config_path = ML_CONFIG_DIR / "ml_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("✓ ML configuration saved")
print(f"  {config_path}")

✓ ML configuration saved
  C:\OceanF\data\processed\ML\ml_config.json


In [59]:
# Verify configuration file

with open(config_path, "r", encoding="utf-8") as f:
    config_check = json.load(f)

print("✓ Configuration loaded successfully")
print(f"Features : {config_check['input_features']}")
print(f"Depths   : {config_check['target_depths_m']}")
print(f"Split    : {config_check['split_strategy']}")
print(f"File     : {config_path}")

✓ Configuration loaded successfully
Features : ['sst', 'sss', 'sla', 'uo', 'vo', 'u_wind', 'v_wind']
Depths   : [0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000]
Split    : chronological
File     : C:\OceanF\data\processed\ML\ml_config.json


In [61]:
import torch
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0+cpu
